In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully")

Libraries imported successfully


In [4]:
import os

os.listdir("../data")

['paysim_transactions.csv.csv']

In [5]:
df = pd.read_csv("../data/paysim_transactions.csv.csv")

df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.shape

(1048575, 11)

In [7]:
df.columns


Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   step            1048575 non-null  int64  
 1   type            1048575 non-null  str    
 2   amount          1048575 non-null  float64
 3   nameOrig        1048575 non-null  str    
 4   oldbalanceOrg   1048575 non-null  float64
 5   newbalanceOrig  1048575 non-null  float64
 6   nameDest        1048575 non-null  str    
 7   oldbalanceDest  1048575 non-null  float64
 8   newbalanceDest  1048575 non-null  float64
 9   isFraud         1048575 non-null  int64  
 10  isFlaggedFraud  1048575 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 88.0 MB


In [9]:
df["isFraud"].value_counts()


isFraud
0    1047433
1       1142
Name: count, dtype: int64

In [10]:
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.89109
1     0.10891
Name: proportion, dtype: float64

In [11]:
df["type"].value_counts()

type
CASH_OUT    373641
PAYMENT     353873
CASH_IN     227130
TRANSFER     86753
DEBIT         7178
Name: count, dtype: int64

In [12]:
df.groupby("type")["isFraud"].sum().sort_values(ascending=False)

type
CASH_OUT    578
TRANSFER    564
CASH_IN       0
DEBIT         0
PAYMENT       0
Name: isFraud, dtype: int64

In [13]:
fraud_rate_by_type = df.groupby("type")["isFraud"].mean().sort_values(ascending=False) * 100
fraud_rate_by_type

type
TRANSFER    0.650122
CASH_OUT    0.154694
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64

In [14]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1047433.0,1.575397e+05,2.541883e+05,0.1,12134.87,76214.97,2.134928e+05,6419835.27
1,1142.0,1.192629e+06,2.030599e+06,119.0,86070.17,353179.45,1.248759e+06,10000000.00


In [15]:
df["origin_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["dest_balance_error"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

df[[
    "oldbalanceOrg", 
    "amount", 
    "newbalanceOrig", 
    "origin_balance_error",
    "oldbalanceDest", 
    "newbalanceDest", 
    "dest_balance_error"
]].head()

,oldbalanceOrg,amount,newbalanceOrig,origin_balance_error,oldbalanceDest,newbalanceDest,dest_balance_error
0,170136.0,9839.64,160296.36,0.0,0.0,0.0,9839.64
1,21249.0,1864.28,19384.72,0.0,0.0,0.0,1864.28
2,181.0,181.00,0.00,0.0,0.0,0.0,181.00
3,181.0,181.00,0.00,0.0,21182.0,0.0,21363.00
4,41554.0,11668.14,29885.86,0.0,0.0,0.0,11668.14


In [16]:
df.groupby("isFraud")[["origin_balance_error", "dest_balance_error"]].describe()

origin_balance_error                                            \
                       count           mean            std         min   
isFraud                                                                  
0                  1047433.0 -178652.254665  281321.954703 -6388051.27   
1                     1142.0   -7937.031935   85459.668523 -1933920.80   

                                                dest_balance_error  \
               25%       50%      75%       max              count   
isFraud                                                              
0       -251045.04 -67828.35 -1580.68  88024.73          1047433.0   
1             0.00      0.00     0.00  38991.68             1142.0   

                                                                             \
                  mean           std          min  25%       50%        75%   
isFraud                                                                       
0         22034.932456  4.293109e+05 -35095923.44  0.0  4413.060   32076.27   
1        567554.586025  1.593695e+06  -8875516.29  0.0  9439.945  353246.50   

                      
                 max  
isFraud               
0        13230407.77  
1        10000000.00

In [17]:
df.groupby("isFraud")[["origin_balance_error", "dest_balance_error"]].mean()

,origin_balance_error,dest_balance_error
isFraud,,
0,-178652.254665,22034.932456
1,-7937.031935,567554.586025


In [18]:
df["is_high_risk_type"] = df["type"].isin(["TRANSFER", "CASH_OUT"]).astype(int)

df[["type", "is_high_risk_type", "isFraud"]].head(10)

,type,is_high_risk_type,isFraud
0,PAYMENT,0,0
1,PAYMENT,0,0
2,TRANSFER,1,1
3,CASH_OUT,1,1
4,PAYMENT,0,0
5,PAYMENT,0,0
6,PAYMENT,0,0
7,PAYMENT,0,0
8,PAYMENT,0,0
9,DEBIT,0,0


In [19]:
df.groupby("is_high_risk_type")["isFraud"].sum()

is_high_risk_type
0       0
1    1142
Name: isFraud, dtype: int64

In [20]:
df["is_high_risk_type"].value_counts()

is_high_risk_type
0    588181
1    460394
Name: count, dtype: int64

In [21]:
df.groupby("is_high_risk_type")["isFraud"].mean() * 100

is_high_risk_type
0    0.000000
1    0.248048
Name: isFraud, dtype: float64

In [22]:
df["sender_txn_count"] = df["nameOrig"].map(df["nameOrig"].value_counts())
df["receiver_txn_count"] = df["nameDest"].map(df["nameDest"].value_counts())

df[["nameOrig", "sender_txn_count", "nameDest", "receiver_txn_count", "isFraud"]].head()

,nameOrig,sender_txn_count,nameDest,receiver_txn_count,isFraud
0,C1231006815,1,M1979787155,1,0
1,C1666544295,1,M2044282225,1,0
2,C1305486145,1,C553264065,26,1
3,C840083671,1,C38997010,27,1
4,C2048537720,1,M1230701703,1,0


In [23]:
df.groupby("isFraud")[["sender_txn_count", "receiver_txn_count"]].mean()

,sender_txn_count,receiver_txn_count
isFraud,,
0,1.000493,10.145925
1,1.000000,7.495622


In [24]:
df["sender_emptied_account"] = (df["newbalanceOrig"] == 0).astype(int)

df.groupby("isFraud")["sender_emptied_account"].mean() * 100

isFraud
0    55.291556
1    99.211909
Name: sender_emptied_account, dtype: float64

In [25]:
amount_95 = df["amount"].quantile(0.95)

amount_95

np.float64(519460.35099999997)

In [26]:
df["is_large_amount"] = (df["amount"] > amount_95).astype(int)

df.groupby("isFraud")["is_large_amount"].mean() * 100

isFraud
0     4.961272
1    40.542907
Name: is_large_amount, dtype: float64

In [27]:
df["risk_score_v1"] = (
    df["is_high_risk_type"] * 20 +
    df["is_large_amount"] * 25 +
    df["sender_emptied_account"] * 25 +
    (df["dest_balance_error"].abs() > df["dest_balance_error"].abs().quantile(0.95)).astype(int) * 20 +
    (df["receiver_txn_count"] > df["receiver_txn_count"].quantile(0.95)).astype(int) * 10
)

df[["type", "amount", "isFraud", "risk_score_v1"]].head(10)

,type,amount,isFraud,risk_score_v1
0,PAYMENT,9839.64,0,0
1,PAYMENT,1864.28,0,0
2,TRANSFER,181.00,1,45
3,CASH_OUT,181.00,1,45
4,PAYMENT,11668.14,0,0
5,PAYMENT,7817.71,0,0
6,PAYMENT,7107.77,0,0
7,PAYMENT,7861.64,0,0
8,PAYMENT,4024.36,0,25
9,DEBIT,5337.77,0,0


In [28]:
df.groupby("isFraud")["risk_score_v1"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1047433.0,25.283087,22.526903,0.0,0.0,25.0,45.0,100.0
1,1142.0,59.316988,17.971573,20.0,45.0,45.0,70.0,100.0


In [29]:
df["rule_alert_v1"] = (df["risk_score_v1"] >= 45).astype(int)

pd.crosstab(df["isFraud"], df["rule_alert_v1"], rownames=["Actual Fraud"], colnames=["Rule Alert"])

Rule Alert,0,1
Actual Fraud,,
0,633067,414366
1,1,1141


In [30]:
true_frauds = 1142
caught_frauds = 1141
false_positives = 414366

recall = caught_frauds / true_frauds
precision = caught_frauds / (caught_frauds + false_positives)

print("Rule Recall:", recall)
print("Rule Precision:", precision)

Rule Recall: 0.999124343257443
Rule Precision: 0.002746042786282782


In [31]:
df.to_csv("../data/frix_features_day1.csv", index=False)

print("Day 1 feature dataset saved successfully")

Day 1 feature dataset saved successfully


In [32]:
import os

os.listdir("../data")

['frix_features_day1.csv', 'paysim_transactions.csv.csv']